# 9.13 · 迁移学习 / Transfer Learning

> **课程定位 / Where this fits**
> 第 13 课，**Part 9 · 深度学习基础**。
> Lesson 13, **Part 9 · Deep Learning Foundations**.
>
> 从零训练一个深度网络需要海量数据和算力。但**别人已经在 ImageNet/大语料上训练好了强大的模型**——我们可以**拿来复用**：保留它学到的通用特征，只在自己的小数据上调整最后几层。这就是**迁移学习**，是实战中**最实用、最高性价比**的技巧，尤其当你数据少的时候。
> Training a deep net from scratch needs massive data and compute. But **others already trained powerful models on ImageNet/large corpora** — we can **reuse them**: keep their general features and adapt only the last layers on our small data. This is **transfer learning**, the most practical, cost-effective trick in practice, especially with little data.
>
> 💼 **实战/面试视角**："特征提取 vs 微调 / 冻结哪些层 / 数据少时怎么办" 是 CV/NLP 岗高频，也是实际项目第一选择。
> 💼 **Practical/interview angle:** "feature extraction vs fine-tuning / which layers to freeze / what to do with little data" — frequent for CV/NLP roles and the default first choice in real projects.

> 📐 **符号约定 / Notation**
> - backbone / 主干 —— 预训练模型去掉最后分类头的特征提取部分 / pretrained model minus its head
> - head / 头 —— 针对新任务新加的输出层 / new output layer for the new task

> 💡 **面试相关 / Interview-relevant**
> - "特征提取 vs 微调的区别"（出镜率 ★★★★★）
> - "什么时候冻结 backbone, 什么时候微调"（★★★★，看数据量/相似度）
> - "微调时为什么用更小的学习率"（★★★★）
> - "为什么迁移学习对小数据特别有效"（★★★★）

---

## 学习目标 / Learning Objectives
1. 理解迁移学习为什么有效（特征可复用）。
   Understand why transfer learning works (reusable features).
2. 区分**特征提取(冻结)** 与 **微调(fine-tuning)**。
   Distinguish feature extraction (freeze) from fine-tuning.
3. 亲手做一个迁移实验，对比"从零训练 vs 迁移"。
   Run a transfer experiment, comparing scratch vs transfer.
4. 知道何时冻结、何时微调、用多大学习率。
   Know when to freeze, when to fine-tune, what LR to use.
5. 了解 torchvision 加载预训练模型的实战写法。
   Know the torchvision pretrained-model API for real use.

## 目录 / TOC
1. [为什么迁移学习有效 ⭐](#1)
2. [准备：预训练一个"源任务"模型 ⭐](#2)
3. [三种策略对比：从零 vs 特征提取 vs 微调 ⭐](#3)
4. [实战写法 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 为什么迁移学习有效 ⭐ / Why Transfer Learning Works

深度网络是**逐层提取特征**的：浅层学到的是**通用、低级**的特征（图像里的边缘、纹理、颜色；文本里的词法、句法），深层才是**任务专属、高级**的特征（"这是猫脸"）。
Deep nets extract features **layer by layer**: shallow layers learn **general, low-level** features (edges, textures, colors in images; morphology, syntax in text), deep layers learn **task-specific, high-level** ones ("this is a cat's face").

**关键洞察**：那些浅层的通用特征**几乎对所有相关任务都有用**。所以与其在小数据上从零学这些特征（很难、容易过拟合），不如**直接借用大模型已经学好的特征**，只把最后专属的部分换成自己的。
**Key insight:** those shallow general features are **useful for almost all related tasks**. So instead of learning them from scratch on small data (hard, overfits), **borrow the features a big model already learned** and just swap the final task-specific part for your own.

**结论**：数据越少，迁移学习的优势越大。这也是为什么实际项目里，**几乎从不从零训练**，而是从预训练模型开始。
**Takeaway:** the less data you have, the bigger the win. That's why real projects **almost never train from scratch** — they start from a pretrained model.


<a id="2"></a>
## 2. 准备：预训练一个"源任务"模型 ⭐ / Pretrain a "Source Task" Model

真实迁移学习会下载 ImageNet 预训练权重（需联网）。为了**离线、自包含**地讲清原理，我们自己造一个迁移场景（一种很常见的迁移：**领域偏移/数据稀缺**）：
Real transfer learning downloads ImageNet weights (needs internet). To explain the principle **offline and self-contained**, we build our own transfer scenario (a very common kind: **domain shift / data scarcity**):
- **源任务(source)**：用**充足的干净数据**训练一个网络识别数字 0~9。它的 backbone 学到识别数字笔画的通用特征。
  **Source task:** train a net on **plenty of clean data** to recognize digits 0–9. Its backbone learns general digit-stroke features.
- **目标任务(target)**：识别**带强噪声**的同样数字，但**只给很少的样本**（模拟现实：真实数据又脏又少）。
  **Target task:** recognize the **heavily noisy** version of the same digits, but with **very few samples** (mimicking reality: real data is dirty and scarce).

直觉：干净数据上学到的"数字长什么样"的特征，应该能帮我们在带噪声的少量数据上快速上手，而不必从零重学。先训练并保存源任务的 backbone。
Intuition: the "what a digit looks like" features learned on clean data should help us get going fast on the noisy few-shot data, without relearning from scratch. First train and save the source backbone.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import copy, torch, torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")

digits = load_digits(); X = digits.data/16.0; y = digits.target
ce = nn.CrossEntropyLoss()
# 源任务: 干净数字 0~9, 充足数据 / source task: clean digits 0-9, plenty of data
X_src, X_pool, y_src, y_pool = train_test_split(X, y, test_size=0.4, stratify=y, random_state=0)

# 网络拆成 backbone(通用特征) + head(分类头) / split into backbone + head
def make_backbone():
    return nn.Sequential(nn.Linear(64,128), nn.ReLU(), nn.Linear(128,64), nn.ReLU())  # 特征提取 / feature extractor
torch.manual_seed(0)
backbone = make_backbone()
src_model = nn.Sequential(backbone, nn.Linear(64, 10))     # 源任务头: 10 类 / source head: 10 classes
opt = torch.optim.Adam(src_model.parameters(), lr=1e-3)
Xst = torch.tensor(X_src, dtype=torch.float32); yst = torch.tensor(y_src)
for _ in range(150):
    opt.zero_grad(); ce(src_model(Xst), yst).backward(); opt.step()
acc_src = (src_model(Xst).argmax(1)==yst).float().mean()
print(f"源任务(干净数字, 充足数据)训练完成, 训练准确率 = {acc_src:.3f}")
print("backbone 已学到识别数字笔画的通用特征 → 下一步迁移到'带噪声的少量数据'任务")
# 保存预训练好的 backbone 权重 / save pretrained backbone weights
pretrained_state = copy.deepcopy(backbone.state_dict())


<a id="3"></a>
## 3. 三种策略对比：从零 vs 特征提取 vs 微调 ⭐ / Scratch vs Feature Extraction vs Fine-tuning

现在做目标任务（识别带噪声的数字），**每类只用很少的样本**，对比三种做法：
Now the target task (recognize noisy digits) with **very few samples per class**, comparing three approaches:

**① 从零训练(scratch)**：随机初始化整个网络，只在这点小数据上训。基线——数据太少，通常**过拟合、效果差**。
**① From scratch:** randomly init the whole net, train only on the tiny data. Baseline — too little data, usually **overfits, poor**.

**② 特征提取(feature extraction，冻结 backbone)**：**加载预训练 backbone 并冻结它**（不更新），只训练一个新的分类头。最快、最省、最适合数据极少时。
**② Feature extraction (freeze backbone):** **load the pretrained backbone and freeze it** (no updates), train only a new head. Fastest, cheapest, best when data is very scarce.

**③ 微调(fine-tuning)**：加载预训练 backbone，**先冻结只训新头(热身)，再解冻、用很小的学习率连 backbone 一起训**，让通用特征也微微适应新任务。数据稍多或领域差异较大时上限更高。
**③ Fine-tuning:** load the pretrained backbone, **first freeze and train only the new head (warm-up), then unfreeze and train the backbone too with a very small LR**, letting general features adapt. Higher ceiling with a bit more data or a larger domain gap.

**两个关键细节（面试常考）**：
**Two key details (interview favorites):**
- **为什么先热身头再解冻**：新头是随机初始化的，一上来就解冻，随机头产生的巨大梯度会**瞬间冲毁**预训练好的 backbone。先用冻结状态把头训好，再温和地微调。
  **Why warm up the head before unfreezing:** the new head is random; unfreezing immediately lets its huge gradients **wreck** the pretrained backbone. Train the head first (frozen), then fine-tune gently.
- **为什么微调用很小的 LR**：预训练权重已经很好，大 LR 会**破坏(灾难性遗忘)**这些宝贵特征；小 LR 只做温和调整。
  **Why a tiny LR for fine-tuning:** pretrained weights are already good; a large LR causes **catastrophic forgetting**; a small LR adjusts gently.


In [ ]:
# 目标任务: 给同样的数字加强噪声, 每类只取 8 个训练样本 / target: noisy digits, 8 samples/class
rng = np.random.RandomState(0)
def add_noise(A): return np.clip(A + rng.normal(0, 0.4, A.shape), 0, 1)   # 加高斯噪声并截到[0,1] / add Gaussian noise
few_idx = np.concatenate([np.where(y_pool == c)[0][:8] for c in range(10)])  # 每类前8个 → 80 样本 / 8 per class
eval_idx = np.setdiff1d(np.arange(len(y_pool)), few_idx)                  # 其余作测试 / rest for test
Xtt = torch.tensor(add_noise(X_pool[few_idx]), dtype=torch.float32); ytt = torch.tensor(y_pool[few_idx])
Xte = torch.tensor(add_noise(X_pool[eval_idx]), dtype=torch.float32); yte = torch.tensor(y_pool[eval_idx])

def build(load_pretrained, freeze):
    torch.manual_seed(1)
    bb = make_backbone()
    if load_pretrained: bb.load_state_dict(pretrained_state)   # 复用源任务学到的特征 / reuse source features
    if freeze:
        for p in bb.parameters(): p.requires_grad = False      # 冻结: 不更新 backbone / freeze backbone
    return bb, nn.Sequential(bb, nn.Linear(64, 10))            # backbone + 新头 / + fresh head

def fit(model, lr, epochs):
    params = [p for p in model.parameters() if p.requires_grad] # 只优化未冻结的参数 / only trainable params
    opt = torch.optim.Adam(params, lr=lr)
    for _ in range(epochs):
        model.train(); opt.zero_grad(); ce(model(Xtt), ytt).backward(); opt.step()

def evaluate(model): return (model(Xte).argmax(1)==yte).float().mean().item()

# ① 从零训练 / from scratch
_, m_scratch = build(load_pretrained=False, freeze=False); fit(m_scratch, 1e-3, 150)
acc_scratch = evaluate(m_scratch)
# ② 特征提取: 冻结 backbone, 只训头 / feature extraction
_, m_feat = build(load_pretrained=True, freeze=True); fit(m_feat, 1e-3, 150)
acc_feat = evaluate(m_feat)
# ③ 微调: 先冻结热身头, 再解冻用很小LR微调 / fine-tuning: warm up head, then unfreeze with tiny LR
bb_ft, m_ft = build(load_pretrained=True, freeze=True)
fit(m_ft, 1e-3, 120)                                          # 阶段1: 热身新头 / phase1: warm up head
for p in bb_ft.parameters(): p.requires_grad = True          # 解冻 backbone / unfreeze
fit(m_ft, 1e-4, 60)                                          # 阶段2: 很小LR微调全网 / phase2: tiny-LR fine-tune
acc_finetune = evaluate(m_ft)

fig, ax = plt.subplots(figsize=(7,4))
names = ["从零训练\nscratch", "特征提取(冻结)\nfeature extract", "微调(两阶段)\nfine-tune"]
accs = [acc_scratch, acc_feat, acc_finetune]
bars = ax.bar(names, accs, color=["#bbb","#4c9","#39c"])
for b,a in zip(bars,accs): ax.text(b.get_x()+b.get_width()/2, a+0.01, f"{a:.3f}", ha="center")
ax.set_ylabel("目标任务 test 准确率"); ax.set_ylim(0,1); ax.set_title("只有80个噪声样本时: 迁移学习 > 从零训练")
plt.tight_layout(); plt.show()
print(f"从零训练:        {acc_scratch:.3f}  (数据太少, 难学好)")
print(f"特征提取(冻结):  {acc_feat:.3f}  (复用预训练特征, 只训新头)")
print(f"微调(两阶段):    {acc_finetune:.3f}  (热身头→解冻小LR微调, 上限更高)")
print("结论: 数据少时, 迁移学习显著优于从零训练; 复用通用特征是关键")


<a id="4"></a>
## 4. 实战写法 + 小结 ⭐ / Practical Code & Summary

真实项目里用 **torchvision**（图像）或 **Hugging Face transformers**（文本）加载预训练模型。下面是 torchvision 的标准套路（伪代码，需联网下权重）：
Real projects load pretrained models via **torchvision** (images) or **Hugging Face transformers** (text). Standard torchvision pattern (pseudo-code, needs internet for weights):


In [ ]:
torchvision_template = """
import torch, torch.nn as nn
from torchvision import models

# 1) 加载在 ImageNet 上预训练好的 ResNet-18 / load ImageNet-pretrained ResNet-18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 2) 特征提取: 冻结所有 backbone 参数 / feature extraction: freeze backbone
for p in model.parameters():
    p.requires_grad = False

# 3) 把最后的分类头换成自己的(比如 10 类) / replace the final head with your own
model.fc = nn.Linear(model.fc.in_features, 10)   # 新头默认 requires_grad=True / new head is trainable

# 4) 只优化新头(特征提取); 若要微调则不冻结并用很小的 LR
opt = torch.optim.Adam(model.fc.parameters(), lr=1e-3)   # 特征提取 / feature extraction
# 微调版: opt = torch.optim.Adam(model.parameters(), lr=1e-5)  # fine-tune with tiny LR

# 注意: 输入要按预训练时的方式预处理(尺寸/归一化均值方差)
# Note: preprocess inputs as during pretraining (size / normalization mean-std)
"""
print("torchvision 迁移学习 4 步: 加载预训练 → 冻结 backbone → 换分类头 → 训练(特征提取或微调)")
print("文本任务同理: Hugging Face AutoModel.from_pretrained(...) + 换/加任务头")
print("\n怎么选(经验法则):")
print("  数据很少 + 任务相似  → 特征提取(冻结 backbone, 只训头), 最省最稳")
print("  数据较多 或 任务差异大 → 微调(解冻, 用很小 LR), 效果上限更高")
print("  数据极多                → 也可考虑从零训练, 但仍常从预训练起步更快")


```
为什么有效: 浅层学通用特征(边缘/纹理/词法), 几乎所有相关任务都能复用; 数据越少优势越大
特征提取(冻结): 加载预训练backbone并冻结, 只训新分类头; 最快最省, 适合数据极少
微调(fine-tune): 连backbone一起训但用很小LR(防灾难性遗忘); 数据较多/任务差异大时更好
怎么选: 数据少+相似→特征提取; 数据多/差异大→微调; 输入要按预训练方式预处理
实战: torchvision(图像)/HuggingFace(文本) 加载预训练 → 冻结 → 换头 → 训练
原则: 实际项目几乎从不从零训练, 从预训练模型起步
```

### 💡 面试速查 / Interview cheat-sheet
1. **为什么有效**: 浅层通用特征可复用, 数据越少优势越大。
   Why it works: shallow general features are reusable; bigger win with less data.
2. **特征提取**: 冻结 backbone 只训头, 最省, 适合数据极少。
   Feature extraction: freeze backbone, train head only, cheapest, for scarce data.
3. **微调**: 解冻 + 很小 LR, 防灾难性遗忘, 上限更高。
   Fine-tuning: unfreeze + tiny LR, avoids catastrophic forgetting, higher ceiling.
4. **怎么选**: 数据少/相似→冻结; 数据多/差异大→微调。
   How to choose: little/similar → freeze; more/different → fine-tune.
5. **预处理一致**: 输入须按预训练的尺寸/归一化处理。
   Match preprocessing: inputs must use pretraining's size/normalization.

### 下一节 / Next
**9.14 神经网络可视化**——网络是黑盒吗? 可视化权重、激活、特征图、loss landscape、用 t-SNE 看学到的表示, 帮你理解和调试模型。
**9.14 NN Visualization** — is the net a black box? Visualize weights, activations, feature maps, the loss landscape, and learned representations via t-SNE to understand and debug models.
